In [51]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from dig4bio.io import read_raman_file
from sklearn.linear_model import LinearRegression
from dig4bio.constants import FINGERPRINT_COLUMNS, LABEL_COLUMNS
from dig4bio.cv import augmented_cross_validate_sample_folds
from dig4bio.models import create_model_factory, AveragePredictor
from dig4bio.paths import EXPERIMENTS_FOLDER
from dig4bio.utils import create_new_experiment_run
from sklearn.model_selection import cross_validate, GroupKFold
import shutil

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
EXPERIMENT_FOLDER = EXPERIMENTS_FOLDER / "exp_001_baselines"
run_folder = create_new_experiment_run(EXPERIMENT_FOLDER)
shutil.copy2(EXPERIMENT_FOLDER/'notebook.ipynb',run_folder/'notebook.ipynb');

In [53]:
source_df = read_raman_file(level='processed',subfolder='source_grid_fingerprint_linear',name='source_datasets.csv')
source_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
789,7068.815451,7047.547414,6990.219828,6932.892241,6913.309013,6902.579399,6893.482759,6886.155172,6880.412017,6882.987124,...,1592.526316,1579.364706,1540.541176,1541.578947,2.99443,0.74913,0.000000,0.0,1,metrohm
326,792.780000,794.920000,797.060000,799.120000,801.180000,797.320000,793.460000,789.425000,785.390000,791.680000,...,773.240000,776.320000,769.795000,763.270000,10.28120,0.66554,0.000000,0.0,1,anton785
313,753.990000,752.965000,751.940000,750.185000,748.430000,752.495000,756.560000,752.815000,749.070000,753.525000,...,759.990000,757.160000,752.380000,747.600000,16.73440,0.00000,0.022216,1.0,0,anton785
637,1118.354100,1117.947133,1117.130033,1114.996700,1111.553633,1107.826000,1105.046600,1104.020567,1104.564333,1105.873100,...,248.771840,249.369919,250.259767,249.023441,0.00000,0.00000,1.483750,0.0,3,kaiser
1477,2352.373443,2379.558217,2350.224082,2305.359791,2251.013524,2124.996044,2134.164597,2199.006120,2314.090307,2573.057821,...,3101.205060,3098.154343,3087.888871,3079.349022,0.52616,0.00000,1.396560,0.0,1,tec
473,951.320000,950.105000,948.890000,945.330000,941.770000,938.610000,935.450000,932.195000,928.940000,927.800000,...,792.210000,796.440000,789.130000,781.820000,0.52096,0.00000,0.015037,0.0,3,anton785
1330,6158.949185,6161.151272,6139.708246,6099.463290,6082.726332,6108.440075,6125.727944,6089.604838,6040.803023,6040.362825,...,876.273901,874.747384,887.460907,885.504552,0.56078,1.07925,3.139890,0.0,4,mettler
49,3099.970000,3099.465000,3098.960000,3095.605000,3092.250000,3074.685000,3057.120000,3030.570000,3004.020000,2989.175000,...,1168.080000,1168.500000,1152.070000,1135.640000,0.00000,0.00000,0.000000,0.0,0,anton532
705,4962.115880,4982.275862,4930.551724,4878.827586,4846.403433,4818.506438,4814.137931,4835.258621,4852.244635,4847.523605,...,1417.970760,1413.541176,1368.835294,1392.298246,1.93507,1.46523,0.000000,0.0,0,metrohm
1891,25104.782790,25057.135650,25046.508152,25105.920830,25125.230770,24972.209820,24729.235073,24605.981800,24655.770981,24738.015552,...,4135.238632,4126.211931,4145.959413,4187.855441,0.24955,0.08245,3.454390,0.0,0,tornado


In [54]:
transfer_plate_df = read_raman_file(level='processed',subfolder='transfer_grid_fingerprint_linear',name='transfer_plate.csv')
transfer_plate_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
17,9060.1875,9021.416149,9055.577640,9046.35625,8991.98125,8924.639752,8876.400,8881.4,8949.732919,9033.56250,...,1645.900621,1643.416149,1646.83750,1658.08750,1674.434783,1672.1875,sample9,6.713935,1.853940,2.054206
159,9607.1875,9548.683230,9655.515528,9638.64375,9493.01875,9465.391304,9440.600,9385.6,9387.850932,9415.70000,...,1659.894410,1682.254658,1693.11875,1688.74375,1693.670807,1685.3875,sample80,7.881890,1.845940,0.837098
7,9574.0000,9519.633540,9605.968944,9600.69375,9500.06875,9443.670807,9384.600,9304.6,9358.826087,9461.26250,...,1672.024845,1672.645963,1673.00000,1673.00000,1685.888199,1684.2500,sample4,2.038084,0.948045,1.380240
177,9589.5000,9558.807453,9566.881988,9553.54375,9517.91875,9491.335404,9448.150,9359.4,9334.378882,9337.03125,...,1680.919255,1703.900621,1707.05625,1683.93125,1690.310559,1691.0625,sample89,5.719336,0.968393,0.434603
147,9712.9375,9677.341615,9630.757764,9597.50625,9578.13125,9464.888199,9378.675,9391.8,9437.813665,9493.95000,...,1698.223602,1703.813665,1695.44375,1668.56875,1668.124224,1666.2250,sample74,7.080100,1.204923,2.470078
120,9677.2500,9599.652174,9656.173913,9625.58750,9504.33750,9494.869565,9477.350,9403.6,9459.403727,9561.48125,...,1685.422360,1695.981366,1699.58125,1693.95625,1702.279503,1692.4375,sample61,10.074314,0.802522,0.812701
50,8462.4375,8444.689441,8441.583851,8427.44375,8401.81875,8373.018634,8345.350,8321.6,8343.142857,8376.96250,...,1624.795031,1644.670807,1654.11875,1649.74375,1647.453416,1632.3875,sample26,9.048045,1.532648,0.528093
54,9750.5625,9701.894410,9670.838509,9622.53750,9556.28750,9539.602484,9523.250,9492.0,9521.701863,9572.93750,...,1707.726708,1700.894410,1685.17500,1657.67500,1672.590062,1680.6875,sample28,11.664203,0.932017,1.361377
174,9609.1875,9538.888199,9617.770186,9602.87500,9490.37500,9429.906832,9382.775,9353.4,9408.621118,9487.37500,...,1672.422360,1682.981366,1689.80625,1691.68125,1690.968944,1683.4000,sample88,3.367320,0.846338,2.356409
138,9411.8125,9313.913043,9353.043478,9357.68750,9326.43750,9264.403727,9200.550,9141.8,9173.260870,9240.86250,...,1662.341615,1695.881988,1703.71250,1677.46250,1682.795031,1677.8375,sample70,2.083604,1.487110,0.745592


In [55]:
### Linear Sources: Linear regression over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(LinearRegression())

sources_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_lr_errors['devices']))
print('Overall')
display(pd.Series(sources_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-9.814037,-11.112856,-268.714326,-520.361891,-178.036716,-11937.375305,-3.594700,-2182.403212
Na_acetate,-13.009612,-0.868218,-17.183729,-67.974554,-209.363922,-6417.786683,-0.516098,-1015.802629
Mg_SO4,-5.064514,-0.036047,-9.014977,-95.357735,-259.089710,-360.502686,-0.240201,-957.250087


Overall


glucose      -2201.480448
Na_acetate   -1198.156705
Mg_SO4        -277.009218
dtype: float64

In [56]:
### Averages Sources: Analyte averages over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(AveragePredictor())

sources_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_avgs_errors['devices']))
print('Overall')
display(pd.Series(sources_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.007701,-0.008376,-0.004897,-0.004898,0.003036,-0.000517,-0.007763,-0.020530
Na_acetate,-0.027002,-0.022158,-0.053501,0.000620,-0.026815,-0.003065,-0.053188,-0.003543
Mg_SO4,-0.076486,-0.077122,-0.237535,-0.023226,-0.065905,-0.022300,-0.258090,-0.014577


Overall


glucose      -0.003056
Na_acetate   -0.003696
Mg_SO4       -0.020624
dtype: float64

In [57]:
### Linear Target: Linear regression train + test over the target device only ###

model_factory = create_model_factory(LinearRegression())

target_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_lr_errors['devices']))
print('Overall')
display(pd.Series(target_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,0.515546,0.528299,0.882741,-3.019211,0.907405,0.612470,0.517172,0.121881
Na_acetate,-0.505592,0.778544,0.821707,0.482408,0.556746,-2.572058,0.690904,-1.108420
Mg_SO4,0.549873,0.835158,0.847339,0.614719,0.879381,-2.023251,0.694120,-10.158555


Overall


glucose       0.028888
Na_acetate   -0.231723
Mg_SO4       -1.627209
dtype: float64

In [58]:
### Linear Averages: Analyte Averages train + test over the target device only ###

model_factory = create_model_factory(AveragePredictor())

target_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_avgs_errors['devices']))
print('Overall')
display(pd.Series(target_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.034112,-0.073546,-0.008200,-0.035465,-0.053714,-0.018685,-0.021394,-0.008342
Na_acetate,-0.056263,-0.023727,-0.027337,-0.020718,-0.014491,-0.030386,-0.007855,-0.022776
Mg_SO4,-0.036037,-0.067311,-0.020365,-0.039999,-0.043428,-0.017934,-0.026028,-0.034934


Overall


glucose      -0.032043
Na_acetate   -0.008138
Mg_SO4        0.000577
dtype: float64

In [ ]:
model = LinearRegression()

test_df = read_raman_file(name='96_samples',level='processed',subfolder='96samples_grid_fingerprint_linear')

X = transfer_plate_df[FINGERPRINT_COLUMNS]
y = transfer_plate_df[LABEL_COLUMNS]

model.fit(X,y)
test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])

display(test_df)

output_df = (
    test_df.copy()
    .groupby(by='sample')
    .mean()
    .reset_index()
)[LABEL_COLUMNS]

output_df.index.name = 'ID'
output_df = output_df.reset_index()
output_df['ID'] +=1
output_df.to_csv(run_folder/f'{EXPERIMENT_FOLDER.name}_{str(run_folder.name[4:])}.csv',index=False)
output_df

# Public Kaggle score of this: -1.23611

C:\Users\J_Bra\AppData\Local\Temp\ipykernel_32320\90002052.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])


,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
0,9459.1250,9440.062112,9440.683230,9403.02500,9325.52500,9295.440994,9269.600,9239.6,9269.279503,9315.65000,...,1650.819876,1671.316770,1689.71875,1705.34375,1696.142857,1672.2125,sample1,1.461898,0.685178,0.353928
1,9441.6250,9411.422360,9435.645963,9429.93125,9393.05625,9312.714286,9256.075,9274.2,9299.850932,9323.63125,...,1655.919255,1678.900621,1679.10000,1649.10000,1665.136646,1674.9625,sample1,2.351013,0.583927,0.467618
2,8607.1875,8614.714286,8571.857143,8517.23125,8450.35625,8408.726708,8371.900,8339.4,8352.229814,8381.90000,...,1603.565217,1642.695652,1666.07500,1668.57500,1651.987578,1625.0000,sample2,5.581757,1.732026,1.431091
3,8560.0625,8553.944099,8543.385093,8527.58750,8506.33750,8434.900621,8377.950,8374.2,8410.590062,8458.28125,...,1609.695652,1627.086957,1630.55000,1615.55000,1632.074534,1635.8375,sample2,6.211434,1.766747,1.229222
4,9432.6875,9347.453416,9321.987578,9321.55625,9347.18125,9274.819876,9216.275,9224.4,9251.919255,9279.65625,...,1677.118012,1705.068323,1704.33750,1665.58750,1693.024845,1707.9500,sample3,4.254881,0.494294,0.999877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,9424.8750,9449.583851,9415.422360,9380.85000,9345.85000,9276.770186,9215.950,9187.2,9226.993789,9288.96875,...,1638.571429,1652.857143,1652.13125,1631.50625,1642.950311,1645.8625,sample94,3.863245,0.210255,0.701643
188,9286.9375,9219.875776,9218.633540,9176.35000,9091.35000,9072.602484,9057.475,9030.6,9054.521739,9094.01250,...,1625.894410,1648.254658,1663.95625,1670.83125,1652.925466,1624.9625,sample95,2.050828,1.525044,0.744649
189,9425.3750,9358.006211,9348.068323,9301.35000,9216.35000,9203.683230,9197.850,9186.6,9245.931677,9329.59375,...,1638.242236,1669.298137,1678.40000,1658.40000,1670.981366,1669.5000,sample95,1.719697,1.459425,0.542773
190,9433.0625,9367.043478,9297.478261,9233.82500,9176.32500,9134.068323,9090.300,9037.8,9117.720497,9246.71875,...,1609.689441,1651.925466,1668.74375,1651.86875,1650.546584,1638.9375,sample96,4.173476,1.753767,0.483940


,ID,glucose,Na_acetate,Mg_SO4
0,1,1.906455,0.634553,0.410773
1,2,9.145176,0.833975,0.326851
2,3,4.515469,1.227353,1.198945
3,4,0.097181,0.947006,0.940205
4,5,0.501357,1.248227,1.221857
...,...,...,...,...
91,92,0.654263,1.251363,1.186624
92,93,4.352456,0.247024,0.550650
93,94,3.855683,0.297089,0.666311
94,95,1.885262,1.492234,0.643711


In [60]:
comparison = pd.DataFrame(
    (sources_avgs_errors['total'],
    sources_lr_errors['total'],
    target_avgs_errors['total'],
    target_lr_errors['total']),
).T

comparison.columns = ['Averages Sources', 'Linear Sources', 'Averages Target', 'Linear Target']
comparison

,Averages Sources,Linear Sources,Averages Target,Linear Target
glucose,-0.003056,-2201.480448,-0.032043,0.028888
Na_acetate,-0.003696,-1198.156705,-0.008138,-0.231723
Mg_SO4,-0.020624,-277.009218,0.000577,-1.627209


In [61]:
comparison.mean(axis=0)

Averages Sources      -0.009125
Linear Sources     -1225.548790
Averages Target       -0.013202
Linear Target         -0.610015
dtype: float64